# Perfilamiento del dataset 04_banco_minorista

Cuaderno de apoyo para la sustentación: muestra paso a paso la comprensión de los datos que después automatiza `src/pipeline.py`. Ejecútelo desde la carpeta `notebooks/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src.config import ruta

crudo = pd.read_csv(ruta('raw'), encoding='utf-8-sig')  # utf-8-sig elimina el BOM
crudo.shape

## 1. Unidad de observación
El id de cliente se repite: cada fila es una obligación de un cliente en una fecha de corte.

In [ ]:
print('Clientes únicos:', crudo['id_cliente_anonimo'].nunique())
crudo.groupby('id_cliente_anonimo').size().describe()

## 2. Tipos, faltantes y rangos

In [ ]:
pd.DataFrame({'tipo': crudo.dtypes.astype(str), 'faltantes': crudo.isna().sum(),
              'pct_faltantes': (crudo.isna().mean()*100).round(2), 'unicos': crudo.nunique()})

In [ ]:
crudo.describe().T

## 3. Categorías

In [ ]:
for c in ['region','producto','segmento','canal_origen','estado_obligacion','ingreso_mensual_rango']:
    print(c, crudo[c].value_counts().to_dict(), '\n')

## 4. Duplicados e inconsistencias

In [ ]:
print('Duplicados exactos:', crudo.duplicated().sum())
print('Mismo cliente, fecha y producto:', crudo.duplicated(['id_cliente_anonimo','fecha_corte','producto']).sum())
esperado = pd.cut(crudo['dias_mora'], [-1,0,30,60,10_000], labels=['Al día','Mora 1-30','Mora 31-60','Mora >60']).astype(str)
print('Estado incoherente con días de mora:', (esperado != crudo['estado_obligacion']).sum())
print('Cuentas de ahorro con días de mora:', ((crudo['producto']=='Cuenta de ahorros') & (crudo['dias_mora']>0)).sum())

## 5. Primera señal analítica: el score frente a la mora

In [ ]:
d = crudo[crudo['producto'] != 'Cuenta de ahorros'].copy()
d['vencida_30'] = d['dias_mora'] > 30
d['banda'] = pd.cut(d['score_riesgo'], [0,500,600,700,800,901], right=False)
d.groupby('banda', observed=True).agg(icv_conteo=('vencida_30','mean'), tasa=('tasa_efectiva_anual_pct','mean'))

## 6. Ejecutar el pipeline completo

In [ ]:
from src.pipeline import ejecutar
perfil = ejecutar()
print(Path(ruta('reportes') / 'perfilamiento_datos.md').read_text(encoding='utf-8')[:1500])